# Model Updator — Investigation

In [ ]:
import sys
sys.path.insert(0, '.')

from src.data.istdaten_fetcher import IstDatenFetcher

fetcher = IstDatenFetcher()
print('user_schema :', fetcher.settings.user_schema)
print('source table:', fetcher.settings.istdaten_table)

In [ ]:
# Cell 1 — fetch latest day from the internet
index = fetcher._build_resource_index()
dates = sorted(index)
print(f'Website date range: {dates[0]}  ->  {dates[-1]}')

rows = fetcher.fetch_day(dates[-1])
print(f'Rows downloaded: {len(rows):,}')
rows[:2]

In [ ]:
# Cell 2 — copy shared cluster istdaten into user schema (one-time init)
fetcher.copy_from_shared(force=False)

In [ ]:
# Cell 3 — check last operating_day in the user's copy
from contextlib import closing

target = f"{fetcher.settings.user_schema}.istdaten"
with closing(fetcher.conn.cursor()) as cur:
    cur.execute(f"SELECT MAX(operating_day) AS last_day, COUNT(*) AS total_rows FROM {target}")
    row = cur.fetchone()
    print(f"last_day   : {row[0]}")
    print(f"total_rows : {row[1]:,}")
    print()
    print(f"Website oldest : {dates[0]}")
    print(f"Website newest : {dates[-1]}")